# Intro to virtual brain model testing

This is an introductory chapter to testing and comparing the predictive equivalence of two models using different simulators. It aims to streamline this process and remove the need for a deep understanding of how the models are designed. 

Why is this useful? There are many models and simulators already out there, but these may have some limitations. For example, the concept of multiple connectivities is completely foreign to TVB library. However, software engineers know that untested software is not correctly written software, and as such arises the need to test them.

In this notebook we'll introduce key concepts that are necessary to build a correct test suite. The good news is that there are relatively few steps and we've built up a solid supporting foundation on how to create such a test suite. Should you follow it and implement the tests, your models can be considered equivalent.

## The requirements

Firstly, you need a ground truth model. For these tutorials we've chosen TVB as it is the gold standard. However, we're showing examples using both VBJax and Neurolib, so if you wish or it suits you better, you may use those. Also, if you're testing a more performant model to another one you've written yourself, this guide should be enough to compare even those models, even if that becomes a bit more involved.

The second requirement is for your tested model to be able to do exactly one step of Euler integration. You may ask why Euler and not Heun, which is the much more widely used approach in virtual brain simulations. This is because with Heun the second part of the integration is a "correction step" which gives a more mathematically sound approximation. However, that is the exact reason why it is not suitable for use in this test suite, as it is another calculation which can be inconsistent between the models, while being correct in both of them. Euler provides a clean deterministic calculation.

The last absolutely necessary requirement is the setting of initial conditions. This means values of state variables, and in case you want to test delays in your model, also some sort of history.

The setting of other values, such as dfun parameters, the time step dt, conduction speed, the connectivity matrix, and the tract‑length matrix is optional, but when used they must be configured identically for both models. So at least one of the models is fully configurable, but ideally both.

## The limitations

As with any testing, there is only so much we can test, and for the rest we'll assume that it's working correctly.

In the requirements section, for example, we mentioned Heun integration and why we won't be testing it. However, should you wish to test that Heun, or any other integrator, is correct, you may do so by comparing sufficiently precise Euler integration against Heun results. As the main benefit of Heun is calculation speed, a one time comparison that is resource-intensive seems like a worthwhile investment.

For dfun testing, we can say that in general there are parameters p1, p2, p3... and state variables v1, v2, v3... In general it is not feasible to test all possible values of v for every permutation of p. The ways to deal with this problem are outlined in the dfun tutorial.

Random number generators are also excluded from testing, and the noise test is based on the assumption you're using a sufficiently random number generator.

## The key concepts

Here we'll present key concepts that we're going to be testing, and with these four concepts we can declare two models predictive equivalence. In layman's terms:
- we're going to check whether the mathematical formulas behind the model are correctly implemented
- whether when we use them in a connected network they interact correctly
- whether they interact correctly if the time it takes to affect the system is not instantaneous
- and whether noise, inherent to real world equivalent of these systems is handled correctly

These ideas will be elaborated on in their individual tutorial notebooks.

## Implementation guidelines

This chapter will show you examples of necessary steps prior to the first test, and explain why they are necessary.

### The configuration

If we make the assumption you have a working knowledge of how virtual brain simulators work, and of the model you're testing, then this is the next step you should make. We're going to show here a mostly complete configuration class, but this guide and adjacent software is designed in such a way that you only need to replace the dfun parameters.

In [1]:
import numpy as np
import tvb.simulator.lab as tvbl

class Config:
    def __init__(self, initial_conditions_seed, noise_seed=42):
        # dfun parameters
        self.a = 0.35
        self.w = 0.2

        # settings relevant for connectivity testing
        self.coupling_strength = 1.0
        self.speed = 2.0
        self.history_length = 10

        # settings relevant for noise testing
        self.dt = 0.1
        self.noise = 0.0
        self.noise_seed = noise_seed

        # generic conditions required for model initialization
        self.conn = None
        self.init_cond_rng = np.random.default_rng(seed=initial_conditions_seed)
        self.init_cond = None

    def __config_connectivity(self):
        # Import connectivity and tract length matrices
        # You're encouraged to use real world connectivities instead of this one
        if self.conn is None:
            self.conn = tvbl.connectivity.Connectivity().from_file()
        np.fill_diagonal(self.conn.weights, 0)  # remove self-connections
        self.conn.speed = np.r_[self.speed]
        self.size = self.conn.weights.shape[0]

    def init_config_for_connectivity(self):
        # Will be shown when relevant
        pass

    def init_config_for_delays(self):
        # Will be shown when relevant
        pass

    def init_cond_for_noise(self):
        # Will be shown when relevant
        pass

    def get_good_history_shape(self):
        # Will be shown when relevant
        pass


/home/dj/diplomka/model_testing/.venv/lib/python3.12/site-packages/tvb/datatypes/surfaces.py:60: UserWarning: Geodesic distance module is unavailable; some functionality for surfaces will be unavailable.
  warnings.warn(msg)


### The wrappers

With an object that holds configuration for both models, we need to write a way for the model to accept this configuration. Thus, a wrapper. The second function of the wrapper is to return results in a reasonable manner, althogh you may `np.reshape` them during the test if you'd like.

Notably, the `config.init_cond` is none for the time being, as different tests make use of different initial conditions.

As you can see, the wrapper is necessary, as even TVB simulators are configured with numbers in some places and arrays of numbers in others. And the return value of `sim.run` is also trimmed.

In [ ]:
import numpy as np
from tvb.simulator import simulator, coupling
from tvb.simulator.integrators import EulerStochastic
from tvb.simulator.monitors import Raw
from tvb.simulator.models.oscillator import SupHopf
import tvb.simulator.lab as tvbl


class TvbModel:
    def __init__(self, config: Config):
        self.config = config
        self._configure_sim()

    def _configure_sim(self):
        self.sim = simulator.Simulator(
            connectivity=self.config.conn,
            model=SupHopf(a=np.r_[self.config.a], omega=np.r_[self.config.w]),
            integrator=EulerStochastic(
                dt=self.config.dt,
                noise=tvbl.noise.Additive(
                    nsig=np.r_[self.config.noise],
                    noise_seed=self.config.noise_seed,
                ),
            ),
            initial_conditions=self.config.init_cond,
            conduction_speed=self.config.speed,
            monitors=[Raw()],
            simulation_length=self.config.dt, # run a single Euler step
            coupling=coupling.Scaling(a=np.r_[self.config.coupling_strength]),
        )
        self.sim.configure()

    def run(self):
        # Return only relevant data from the simulation
        return self.sim.run()[0][1]
